# ScamSense — Notebook 04: LangGraph Agentic Prediction Pipeline

This notebook implements the ScamSense **agentic pipeline** using **LangGraph**.
Three specialised agents communicate through a shared state graph, with conditional
routing that decides whether to run the full scam analysis or exit early for safe messages.

| Agent | Role | Method |
|---|---|---|
| **Detection Agent** | Classify Scam / Ham + detect language | Fine-tuned XLM-RoBERTa |
| **Explanation Agent** | SHAP token attribution + **RAG over SPF knowledge base** | SHAP + FAISS |
| **Risk Agent** | Scam type + risk score + SPF 2025 statistics | Rule-based SPF taxonomy |

**Why LangGraph?**  
LangGraph enforces a proper agent architecture: each agent is a node, state is passed
explicitly between nodes, and conditional edges allow the graph to branch — for example,
bypassing the Explanation and Risk agents entirely when a message is clearly safe.

```
User Message
      │
      ▼
┌─────────────────┐
│ Detection Agent │  ← XLM-RoBERTa
└────────┬────────┘
         │
    Ham (≥80%)?
    ┌────┴─────┐
   Yes         No
    │          │
    ▼          ▼
┌────────┐  ┌──────────────────┐
│  Safe  │  │ Explanation Agent│  ← SHAP + FAISS RAG (SPF corpus)
│  Exit  │  └────────┬─────────┘
└────────┘           │
                     ▼
              ┌─────────────┐
              │  Risk Agent │  ← SPF 2025 taxonomy
              └─────────────┘
```



## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
# Run this cell first, then restart the kernel when prompted

packages = [
    "langgraph",
    "langdetect",
    "shap",
    "sentence-transformers==3.0.1",
    "faiss-cpu",
    "numpy<2",
    "pandas",
    "tabulate",
]

import subprocess, sys
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print(f"  ✔ {pkg}")

print("\n✅ All packages installed — restart kernel now")
import os; os.kill(os.getpid(), 9)

  ✔ langgraph
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 24.9 MB/s eta 0:00:00
  ✔ langdetect
  ✔ shap
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.3 MB/s eta 0:00:00
  ✔ sentence-transformers==3.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 83.6 MB/s eta 0:00:00
  ✔ faiss-cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 73.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.

  ✔ numpy<2
  ✔ pandas


## Cell 2 — Imports & Setup

In [1]:
import re, json
from pathlib import Path
from typing import TypedDict, Optional

import numpy as np
import torch
import faiss
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
import shap
from langdetect import detect, LangDetectException
from langgraph.graph import StateGraph, END

# ── Kaggle Secrets ────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    secrets  = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None   

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = Path("/kaggle/working/ScamSense")
RAG_DIR  = BASE_DIR / "rag"
RAG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device  : {DEVICE}")
print(f"Base dir: {BASE_DIR}")
print("✅ Imports complete")

2026-06-20 19:02:23.182949: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781982143.397143     119 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781982143.458080     119 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781982143.962487     119 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781982143.962531     119 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781982143.962534     119 computation_placer.cc:177] computation placer alr

Device  : cpu
Base dir: /kaggle/working/ScamSense
✅ Imports complete


## Cell 3 — Load Fine-tuned XLM-RoBERTa Classifier

In [2]:
HF_MODEL_ID = "Bhoovika/scamsense-xlmroberta"

print(f"Loading tokenizer from {HF_MODEL_ID} ...")
clf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, token=HF_TOKEN)

print("Loading model ...")
clf_model = AutoModelForSequenceClassification.from_pretrained(
    HF_MODEL_ID, token=HF_TOKEN
).to(DEVICE)
clf_model.eval()

LABEL_MAP = {0: "ham", 1: "scam"}

def run_classifier(text: str) -> dict:
    """Run XLM-RoBERTa on a single text. Returns label, confidence, scam_prob, logits."""
    inputs = clf_tokenizer(
        text, return_tensors="pt", truncation=True, max_length=128, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    probs    = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_idx = int(probs.argmax())
    return {
        "label"     : LABEL_MAP[pred_idx],
        "confidence": round(float(probs[pred_idx]), 4),
        "scam_prob" : round(float(probs[1]), 4),
        "logits"    : logits.cpu().numpy()[0].tolist(),
    }

# Sanity check
test = run_classifier("You won $10,000! Claim now.")
print(f"\nSanity check → label={test['label']} | confidence={test['confidence']:.1%}")
print("✅ Classifier ready")

Loading tokenizer from Bhoovika/scamsense-xlmroberta ...


tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

Loading model ...


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]


Sanity check → label=scam | confidence=100.0%
✅ Classifier ready


## Cell 4 — SPF 2025 Risk Taxonomy

Rule-based taxonomy derived from the **SPF Annual Scams & Cybercrime Brief 2025**.
Used by the Risk Agent to classify scam type and compute a risk score.

> **Why not load from a PDF?**  
> The SPF report is a government PDF with mixed layouts, tables, and charts — 
> direct PDF parsing produces noisy, unreliable text. Instead the key statistics 
> and advisory text were manually extracted and encoded here as structured data. 
> This is standard practice for RAG systems where source quality matters.
> The same passages are embedded into the FAISS index in Cell 5 for semantic retrieval.


In [3]:
# Source: Singapore Police Force, Annual Scams & Cybercrime Brief 2025 (25 Feb 2026)

SPF_TAXONOMY = {
    "investment": {
        "spf_name"    : "Investment Scam",
        "2025_cases"  : 5462,
        "2025_losses" : 336.2,   # SGD millions
        "avg_loss"    : 61559,
        "risk_tier"   : "CRITICAL",
        "risk_score"  : 95,
        "keywords"    : [
            r"invest", r"return", r"profit", r"forex", r"crypto",
            r"trading", r"guaranteed", r"passive income", r"portfolio",
            r"capital", r"dividend", r"scheme", r"fund", r"roi",
            r"bitcoin", r"ethereum", r"tether", r"usdt", r"wallet",
            r"keuntungan", r"pelaburan",
            r"முதலீட்", r"லாபம்",
            r"投资", r"收益", r"理财", r"赚钱",
        ],
        "advice": (
            "Investment scams caused the HIGHEST losses in Singapore in 2025 — "
            "$336.2 million (SPF 2025). Never transfer money to strangers for "
            "'investments'. Verify at MAS (mas.gov.sg/investor-alert-list). "
            "Report to SPF at 1800-255-0000."
        ),
    },
    "impersonation": {
        "spf_name"    : "Government Officials Impersonation Scam",
        "2025_cases"  : 3363,
        "2025_losses" : 242.9,
        "avg_loss"    : 72229,
        "risk_tier"   : "CRITICAL",
        "risk_score"  : 93,
        "keywords"    : [
            r"police", r"spf", r"cpf", r"ica", r"mas", r"iras", r"hdb",
            r"officer", r"detective", r"warrant", r"arrest", r"investigation",
            r"money laundering", r"safety account", r"court", r"ministry",
            r"government", r"official", r"authority", r"polis",
            r"警察", r"公安", r"调查", r"安全账户",
            r"காவல்", r"அரசு",
        ],
        "advice": (
            "Cases MORE THAN DOUBLED in 2025 (+123.6%), $242.9M lost (SPF 2025). "
            "Singapore government officials will NEVER ask you to transfer money, "
            "disclose banking details, or install unofficial apps. "
            "Hang up and call ScamShield at 1799."
        ),
    },
    "job": {
        "spf_name"    : "Job Scam",
        "2025_cases"  : 5575,
        "2025_losses" : 123.5,
        "avg_loss"    : 22163,
        "risk_tier"   : "HIGH",
        "risk_score"  : 80,
        "keywords"    : [
            r"job", r"work from home", r"part.?time", r"hiring", r"salary",
            r"task", r"commission", r"earn", r"vacancy", r"recruit",
            r"registration fee", r"training fee", r"deposit", r"agent fee",
            r"kerja", r"gaji", r"pendapatan",
            r"வேலை", r"சம்பளம்",
            r"工作", r"兼职", r"佣金", r"招聘",
        ],
        "advice": (
            "Job scams cost Singaporeans $123.5M in 2025 (SPF 2025). "
            "Legitimate employers never ask for upfront fees. "
            "Verify at mom.gov.sg. Report to MOM or call 1800-255-0000."
        ),
    },
    "phishing": {
        "spf_name"    : "Phishing Scam",
        "2025_cases"  : 6264,
        "2025_losses" : 39.9,
        "avg_loss"    : 6384,
        "risk_tier"   : "HIGH",
        "risk_score"  : 78,
        "keywords"    : [
            r"click", r"link", r"verify", r"suspend", r"account", r"login",
            r"password", r"otp", r"credential", r"bank", r"dbs", r"ocbc",
            r"uob", r"posb", r"singpass", r"cpf", r"paynow", r"card",
            r"http", r"www\.", r"\.xyz", r"\.top", r"\.club",
            r"kata laluan", r"akaun",
            r"கணக்கு", r"கடவுச்சொல்",
            r"账户", r"密码", r"验证", r"点击",
        ],
        "advice": (
            "Phishing is the 2nd most common scam in Singapore (6,264 cases, SPF 2025). "
            "Never click unsolicited links or enter OTP/password from an SMS or email. "
            "Go directly to your bank's official website. "
            "Report to report@scamalert.sg."
        ),
    },
    "ecommerce": {
        "spf_name"    : "E-commerce Scam",
        "2025_cases"  : 6703,
        "2025_losses" : 16.7,
        "avg_loss"    : 2503,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 60,
        "keywords"    : [
            r"sell", r"selling", r"buy", r"cheap", r"deal", r"item",
            r"carousell", r"shopee", r"lazada", r"facebook marketplace",
            r"ship", r"delivery", r"transfer first", r"paynow first",
            r"deposit", r"pokemon", r"brand new", r"legit",
            r"jual", r"beli", r"murah",
            r"விற்பனை", r"வாங்க",
            r"出售", r"购买", r"便宜", r"转账",
        ],
        "advice": (
            "E-commerce scams are the most common type in Singapore (6,703 cases, SPF 2025). "
            "Never PayNow before receiving goods. "
            "Meet in person for high-value items or use Carousell protected payment. "
            "Report at go.gov.sg/scamalert."
        ),
    },
    "love": {
        "spf_name"    : "Internet Love Scam",
        "2025_cases"  : 917,
        "2025_losses" : 24.9,
        "avg_loss"    : 27202,
        "risk_tier"   : "HIGH",
        "risk_score"  : 75,
        "keywords"    : [
            r"love", r"relationship", r"dating", r"girlfriend", r"boyfriend",
            r"miss you", r"together", r"meet", r"lonely", r"soulmate",
            r"overseas", r"stranded", r"emergency", r"send money",
            r"cinta", r"sayang",
            r"அன்பு", r"காதல்",
            r"爱", r"恋爱", r"想你",
        ],
        "advice": (
            "Internet love scams averaged $27,202 per victim in 2025 (SPF 2025). "
            "Be wary of online relationships where the person never video calls, "
            "claims to be overseas, and eventually asks for money. "
            "Call ScamShield at 1799."
        ),
    },
    "unknown": {
        "spf_name"    : "Other Scam",
        "2025_cases"  : 9941,
        "2025_losses" : 135.1,
        "avg_loss"    : 13590,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 55,
        "keywords"    : [],
        "advice": (
            "This message shows scam indicators. Do not send money, share personal "
            "details or OTPs, or click unknown links. "
            "Verify via ScamShield app or call 1799."
        ),
    },
}

def classify_scam_type(text: str) -> tuple:
    """Match text against SPF taxonomy keywords. Returns (scam_type, risk_score, risk_tier, advice)."""
    text_lower = text.lower()
    scores = {}
    for stype, info in SPF_TAXONOMY.items():
        if stype == "unknown":
            continue
        hits = sum(1 for kw in info["keywords"] if re.search(kw, text_lower))
        if hits > 0:
            scores[stype] = hits
    if not scores:
        t = SPF_TAXONOMY["unknown"]
        return "unknown", t["risk_score"], t["risk_tier"], t["advice"]
    best = max(scores, key=scores.get)
    t = SPF_TAXONOMY[best]
    return best, t["risk_score"], t["risk_tier"], t["advice"]

# Quick test
print(classify_scam_type("guaranteed 300% returns on crypto investment!"))
print("✅ SPF taxonomy ready")

('investment', 95, 'CRITICAL', "Investment scams caused the HIGHEST losses in Singapore in 2025 — $336.2 million (SPF 2025). Never transfer money to strangers for 'investments'. Verify at MAS (mas.gov.sg/investor-alert-list). Report to SPF at 1800-255-0000.")
✅ SPF taxonomy ready


## Cell 5 — SPF Advisory Corpus + FAISS Index

The SPF corpus contains **30 advisory passages** extracted from the SPF 2025 brief.
Each passage is a self-contained chunk covering one scam type or statistic.

These are embedded using `paraphrase-multilingual-MiniLM-L12-v2` (supports all 5 languages)
and stored in a FAISS index for fast cosine-similarity retrieval.

The index is saved to disk so it only needs to be built once per Kaggle session.


In [4]:
SPF_CORPUS = [
    # ── OVERALL ──────────────────────────────────────────────────────────────
    {"id":"spf_001","scam_type":"overview","topic":"Overall scam situation 2025","source_page":1,
     "text":("In 2025, scam and cybercrime cases decreased by 24.8% to 41,974 cases. "
             "Scam cases fell by 27.6% to 37,308 cases. Total losses fell 17.9% to $913.1 million. "
             "Despite the decrease, the situation remains very concerning.")},
    {"id":"spf_002","scam_type":"overview","topic":"Self-effected transfers 2025","source_page":4,
     "text":("81.8% of scams involved self-effected transfers — scammers manipulated victims into "
             "performing monetary transactions through deception and social engineering, "
             "without gaining direct control of victims' accounts.")},
    {"id":"spf_003","scam_type":"overview","topic":"Cryptocurrency losses 2025","source_page":3,
     "text":("Cryptocurrency losses accounted for $182.2 million (about 20% of total scam losses) "
             "in 2025. Tether, Ethereum, and Bitcoin accounted for 91.7% of cryptocurrency losses. "
             "Crypto transactions are irreversible, making recovery very challenging.")},
    {"id":"spf_004","scam_type":"overview","topic":"Top scam types by cases 2025","source_page":4,
     "text":("Top 5 scam types by cases in 2025: e-commerce (6,703 cases, 18.0%), "
             "phishing (6,264 cases, 16.8%), job scams (5,575 cases, 14.9%), "
             "investment scams (5,462 cases, 14.6%), government impersonation (3,363 cases, 9.0%).")},
    {"id":"spf_005","scam_type":"overview","topic":"Top scam types by losses 2025","source_page":5,
     "text":("Top 5 scam types by losses in 2025: investment scams ($336.2M, 36.8%), "
             "government impersonation ($242.9M, 26.6%), job scams ($123.5M, 13.5%), "
             "phishing ($39.9M, 4.4%), business email compromise ($35.3M, 3.9%).")},
    # ── INVESTMENT ────────────────────────────────────────────────────────────
    {"id":"spf_006","scam_type":"investment","topic":"Investment scam statistics 2025","source_page":7,
     "text":("Investment scams recorded the highest losses in 2025: $336.2 million (+4.8% from 2024). "
             "There were 5,462 cases. Average loss per victim: $61,559 — highest of all scam types.")},
    {"id":"spf_007","scam_type":"investment","topic":"Investment scam tactics — platforms","source_page":18,
     "text":("Victims encountered investment opportunities via Telegram and Facebook chat groups, "
             "online ads, and recommendations from online contacts. They were shown false testimonies "
             "and instructed to transfer money to bank accounts or cryptocurrency wallets.")},
    {"id":"spf_008","scam_type":"investment","topic":"Investment scam — crypto tactics","source_page":18,
     "text":("Investment scams accounted for 38.4% of all crypto-related scam cases in 2025. "
             "Victims were directed to open crypto accounts, fund them from bank accounts, "
             "then transfer cryptocurrency to scammer-controlled wallets.")},
    {"id":"spf_009","scam_type":"investment","topic":"Investment scam — fake apps","source_page":19,
     "text":("Scammers directed victims to download fake investment apps showing fictitious profits. "
             "Victims only discovered the scam when they attempted to withdraw returns and "
             "were asked to pay increasingly large 'fees' or 'taxes'.")},
    # ── IMPERSONATION ─────────────────────────────────────────────────────────
    {"id":"spf_010","scam_type":"impersonation","topic":"Government impersonation statistics 2025","source_page":6,
     "text":("Government impersonation scams more than doubled in 2025 (+123.6% to 3,363 cases). "
             "Losses were $242.9 million (+60.5%). Average loss per victim: $72,229 — highest of all types.")},
    {"id":"spf_011","scam_type":"impersonation","topic":"Government impersonation — bank transfer tactic","source_page":17,
     "text":("91.7% of government impersonation cases: victims received unsolicited calls from scammers "
             "posing as bank representatives, then were transferred to fake government officials who "
             "accused them of money laundering and told them to transfer funds to 'safety accounts'.")},
    {"id":"spf_012","scam_type":"impersonation","topic":"What Singapore government officials will never do","source_page":18,
     "text":("Singapore Government officials will NEVER ask you over a phone call to: transfer money, "
             "disclose banking login details, install apps from unofficial stores, or transfer "
             "your call to the Police. Never hand money or valuables to any unknown person.")},
    {"id":"spf_013","scam_type":"impersonation","topic":"Government impersonation — PayNow and crypto","source_page":17,
     "text":("New 2025 tactics: scammers instructed victims to transfer funds via PayNow to "
             "Payment Service Provider accounts (e.g. YouTrip), or to open new crypto accounts "
             "and transfer cryptocurrency directly to scammer-controlled wallets.")},
    # ── JOB SCAMS ─────────────────────────────────────────────────────────────
    {"id":"spf_014","scam_type":"job","topic":"Job scam statistics 2025","source_page":16,
     "text":("Job scams: 5,575 cases in 2025 (down 38.4% from 2024). Total losses: $123.5 million. "
             "Average loss per victim: $22,163. Third highest by both cases and losses.")},
    {"id":"spf_015","scam_type":"job","topic":"Job scam tactics — upfront fees and fake tasks","source_page":16,
     "text":("Job scammers advertise easy work-from-home jobs with high pay on social media. "
             "Victims are asked to pay registration fees, training fees, or deposits before starting. "
             "Some involve fake online tasks (liking posts, boosting products) with small initial payments "
             "to build trust, before large sums are requested and never returned.")},
    {"id":"spf_016","scam_type":"job","topic":"Job scam — victim demographics","source_page":9,
     "text":("Job scams disproportionately targeted young adults aged 20–29 (19.9% of all victims) "
             "and adults aged 30–49 (17.6% falling prey to job scams). "
             "Legitimate employers do not ask for upfront fees before employment.")},
    # ── PHISHING ──────────────────────────────────────────────────────────────
    {"id":"spf_017","scam_type":"phishing","topic":"Phishing scam statistics 2025","source_page":7,
     "text":("Phishing scams: 6,264 cases in 2025 (down 26.8% from 2024). "
             "Total losses: $39.9 million (down 32.8%). Average loss per victim: $6,384.")},
    {"id":"spf_018","scam_type":"phishing","topic":"Phishing tactics — card details and OTP theft","source_page":7,
     "text":("Phishing scams predominantly involved victims submitting card details and OTPs "
             "on fake websites impersonating banks, government agencies, or e-commerce platforms, "
             "resulting in unauthorised card transactions.")},
    {"id":"spf_019","scam_type":"phishing","topic":"Phishing contact methods","source_page":11,
     "text":("Phishing scammers contact victims via SMS, email, and messaging platforms with links "
             "to fraudulent websites. Major retail banks have phased out SMS OTPs for digital token users. "
             "Never click links in unsolicited messages — go directly to your bank's official app or website.")},
    # ── E-COMMERCE ────────────────────────────────────────────────────────────
    {"id":"spf_020","scam_type":"ecommerce","topic":"E-commerce scam statistics 2025","source_page":6,
     "text":("E-commerce scams: 6,703 cases in 2025 (down 42.5% from 2024, but still the most cases). "
             "Total losses: $16.7 million. Average loss per victim: $2,503.")},
    {"id":"spf_021","scam_type":"ecommerce","topic":"E-commerce scam platforms and payment","source_page":6,
     "text":("Carousell (29.0%) and Facebook Marketplace (22.2%) were the most used platforms. "
             "Victims were asked to pay via PayNow or bank transfer upfront. "
             "They discovered the scam when goods were not delivered and sellers became uncontactable.")},
    {"id":"spf_022","scam_type":"ecommerce","topic":"E-commerce scam — most common items","source_page":6,
     "text":("Pokemon trading cards were the most commonly scammed item (13.6% of e-commerce cases). "
             "Other items: electronics, concert tickets, luxury goods. "
             "Use platforms' protected payment and meet sellers in person for high-value items.")},
    # ── LOVE SCAMS ────────────────────────────────────────────────────────────
    {"id":"spf_023","scam_type":"love","topic":"Internet love scam statistics 2025","source_page":16,
     "text":("Internet love scams: 917 cases in 2025 (+7.6% from 2024). Total losses: $24.9 million. "
             "Average loss per victim: $27,202 — much higher than e-commerce or phishing, "
             "reflecting extended trust-building by scammers.")},
    {"id":"spf_024","scam_type":"love","topic":"Love scam tactics","source_page":16,
     "text":("Love scammers build relationships over weeks or months via dating apps or social media, "
             "posing as overseas professionals. Once trust is established they fabricate emergencies "
             "requiring urgent money transfers. Red flags: refusal to video call, claims of being stranded.")},
    # ── CONTACT METHODS ───────────────────────────────────────────────────────
    {"id":"spf_025","scam_type":"overview","topic":"Online platforms used by scammers 2025","source_page":8,
     "text":("Scammers used online platforms in 84.1% of all cases. Meta platforms: 35.4% of cases. "
             "Top contact methods: social media (10,448 cases), messaging platforms (9,355), "
             "phone calls (5,477), online shopping platforms (3,804).")},
    {"id":"spf_026","scam_type":"overview","topic":"WhatsApp and Telegram most used for scam contact","source_page":8,
     "text":("Among messaging platforms: WhatsApp 53.5%, Telegram 37.9%, Facebook Messenger 4.2%. "
             "Among social media: Facebook 51.9%, TikTok 26.0%, Instagram 14.2% of scam contact cases.")},
    # ── VICTIM PROFILE ────────────────────────────────────────────────────────
    {"id":"spf_027","scam_type":"overview","topic":"Scam victim age profile 2025","source_page":9,
     "text":("85.2% of scam victims were aged below 65. Adults aged 30–49: 36.1% of victims ($22,283 avg loss). "
             "Elderly aged 65+: 14.8% with the highest average loss of $37,053, mainly from investment, "
             "impersonation, and phishing scams.")},
    # ── ANTI-SCAM ACTIONS ─────────────────────────────────────────────────────
    {"id":"spf_028","scam_type":"overview","topic":"ScamShield helpline and reporting channels","source_page":13,
     "text":("When in doubt, call ScamShield Helpline at 1799 or use the ScamShield app. "
             "Report scams to SPF at 1800-255-0000 or police.gov.sg/i-witness. "
             "Report phishing links to report@scamalert.sg. "
             "Check investment legitimacy at MAS Investor Alert List: mas.gov.sg.")},
    {"id":"spf_029","scam_type":"overview","topic":"Anti-scam laws — caning and Protection from Scams Act","source_page":10,
     "text":("From 30 December 2025, caning for scam offences was operationalised (6–24 strokes). "
             "The Protection from Scams Act (1 July 2025) allows SPF to restrict banking facilities "
             "of scam victims who continue to believe scammers despite police engagement.")},
    {"id":"spf_030","scam_type":"overview","topic":"Money mules — do not share accounts or Singpass","source_page":13,
     "text":("Do not allow anyone to use your financial accounts to transfer funds, share Singpass "
             "credentials, or supply local SIM cards. In 2025, 7,000+ money mules were investigated "
             "and 940+ charged. Penalties include substantial prison terms and caning.")},
]

print(f"SPF corpus loaded: {len(SPF_CORPUS)} advisory chunks")
print(f"Scam types covered: {sorted(set(c['scam_type'] for c in SPF_CORPUS))}")

SPF corpus loaded: 30 advisory chunks
Scam types covered: ['ecommerce', 'impersonation', 'investment', 'job', 'love', 'overview', 'phishing']


In [5]:
# ── Build or load FAISS index ─────────────────────────────────────────────────
INDEX_PATH     = RAG_DIR / "spf_faiss.index"
EMBEDDINGS_PATH = RAG_DIR / "spf_embeddings.npy"

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

if INDEX_PATH.exists() and EMBEDDINGS_PATH.exists():
    index             = faiss.read_index(str(INDEX_PATH))
    corpus_embeddings = np.load(str(EMBEDDINGS_PATH))
    print(f"✅ FAISS index loaded from disk ({index.ntotal} vectors)")
else:
    print("Building FAISS index from SPF corpus ...")
    corpus_texts      = [c["text"] for c in SPF_CORPUS]
    corpus_embeddings = embedder.encode(
        corpus_texts,
        normalize_embeddings=True,   
        show_progress_bar=True
    ).astype(np.float32)

    DIM   = corpus_embeddings.shape[1]
    index = faiss.IndexFlatIP(DIM)
    index.add(corpus_embeddings)

    faiss.write_index(index, str(INDEX_PATH))
    np.save(str(EMBEDDINGS_PATH), corpus_embeddings)
    print(f"✅ FAISS index built ({index.ntotal} vectors, dim={DIM})")
    print(f"   Saved → {INDEX_PATH}")

# Quick retrieval test
q = embedder.encode(["phishing bank account OTP"], normalize_embeddings=True).astype(np.float32)
scores, idxs = index.search(q, 3)
print("\nRetrieval test (phishing query):")
for s, i in zip(scores[0], idxs[0]):
    print(f"  [{s:.3f}] {SPF_CORPUS[i]['id']} — {SPF_CORPUS[i]['topic']}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS index from SPF corpus ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS index built (30 vectors, dim=384)
   Saved → /kaggle/working/ScamSense/rag/spf_faiss.index

Retrieval test (phishing query):
  [0.721] spf_019 — Phishing contact methods
  [0.686] spf_018 — Phishing tactics — card details and OTP theft
  [0.505] spf_013 — Government impersonation — PayNow and crypto


## Cell 6 — SHAP Explainer, Language Detector & Agent State Schema

In [6]:
# ── SHAP ─────────────────────────────────────────────────────────────────────
def shap_predict(texts):
    """Wrapper so SHAP can query the classifier. Returns [ham_prob, scam_prob] per text."""
    if isinstance(texts, str):
        texts = [texts]
    inputs = clf_tokenizer(
        list(texts), return_tensors="pt", truncation=True, max_length=128, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

masker    = shap.maskers.Text(clf_tokenizer)
explainer = shap.Explainer(shap_predict, masker, output_names=["ham", "scam"])

def get_top_shap_features(text: str, top_n: int = 5) -> list[dict]:
    """Return top_n tokens most responsible for the scam prediction (by SHAP value)."""
    shap_values = explainer([text])
    token_names = shap_values.data[0]
    token_shaps = shap_values.values[0, :, 1]   # scam class
    pairs = [
        {"token": tok, "shap_value": round(float(val), 4)}
        for tok, val in zip(token_names, token_shaps)
        if tok not in ["", "▁", "<s>", "</s>", "<pad>"]
    ]
    return sorted(pairs, key=lambda x: x["shap_value"], reverse=True)[:top_n]

print("✅ SHAP explainer ready")

# ── Singlish heuristics ───────────────────────────────────────────────────────
SINGLISH_MARKERS = [
    r"\blah\b", r"\bleh\b", r"\bsia\b", r"\blor\b", r"\bwah\b",
    r"\baiyo\b", r"\bsiao\b", r"\bdun\b", r"\bwan\b", r"\bcan or not\b",
    r"\bgot meh\b", r"\bcan leh\b", r"\bone\b",
]
_singlish_re = re.compile("|".join(SINGLISH_MARKERS), re.IGNORECASE)

def detect_language(text: str) -> str:
    hits  = len(_singlish_re.findall(text))
    words = len(text.split())
    if words > 0 and (hits / words) >= 0.08:
        return "singlish"
    try:
        d = detect(text)
        return {"en":"en","ms":"ms","id":"ms","ta":"ta",
                "zh-cn":"zh","zh-tw":"zh","zh":"zh"}.get(d, "en")
    except LangDetectException:
        return "en"

print("✅ Language detector ready")

# ── LangGraph AgentState ──────────────────────────────────────────────────────
class AgentState(TypedDict):
    # Input
    message      : str
    # Detection Agent outputs
    language     : Optional[str]
    label        : Optional[str]
    confidence   : Optional[float]
    scam_prob    : Optional[float]
    # Explanation Agent outputs
    top_features : Optional[list]
    rag_chunks   : Optional[list]   # retrieved SPF passages
    rag_summary  : Optional[str]    # condensed RAG explanation
    # Risk Agent outputs
    scam_type    : Optional[str]
    spf_name     : Optional[str]
    risk_score   : Optional[int]
    risk_tier    : Optional[str]
    advice       : Optional[str]
    # Final combined output
    output       : Optional[dict]

print("✅ AgentState schema defined")

✅ SHAP explainer ready
✅ Language detector ready
✅ AgentState schema defined


## Cell 7 — Define the Three Agent Nodes

Each agent is a **LangGraph node** — a function that receives the full shared state,
does its work, and returns updated state fields.

- **Detection Agent**: XLM-RoBERTa classification + language detection
- **Explanation Agent**: SHAP token attribution + **RAG retrieval from FAISS** (SPF corpus)
- **Risk Agent**: SPF taxonomy matching + risk score + advice


In [7]:
# ── RAG retrieval helper ──────────────────────────────────────────────────────
def retrieve_spf_passages(message: str, scam_type: str, top_k: int = 3) -> list[dict]:
    """
    Retrieve top-k relevant SPF advisory passages for a message.
    Combines scam-type-specific results with general top results, deduplicated.
    """
    q_embed = embedder.encode([message], normalize_embeddings=True).astype(np.float32)

    # Type-specific retrieval (higher precision)
    type_indices = [i for i, c in enumerate(SPF_CORPUS) if c["scam_type"] == scam_type]
    if not type_indices:
        type_indices = list(range(len(SPF_CORPUS)))

    type_embeds = corpus_embeddings[type_indices]
    type_scores = (q_embed @ type_embeds.T)[0]
    top_type    = np.argsort(type_scores)[::-1][:top_k]

    results, seen = [], set()
    for local_i in top_type:
        chunk = SPF_CORPUS[type_indices[local_i]].copy()
        chunk["score"] = round(float(type_scores[local_i]), 4)
        if chunk["id"] not in seen:
            seen.add(chunk["id"])
            results.append(chunk)

    # Add general top result for breadth
    gen_scores, gen_idx = index.search(q_embed, 2)
    for s, i in zip(gen_scores[0], gen_idx[0]):
        chunk = SPF_CORPUS[i].copy()
        chunk["score"] = round(float(s), 4)
        if chunk["id"] not in seen:
            seen.add(chunk["id"])
            results.append(chunk)

    return results[:top_k + 1]


# ─────────────────────────────────────────────────────────────────────────────
# Agent 1: Detection Agent
# ─────────────────────────────────────────────────────────────────────────────
def detection_agent(state: AgentState) -> AgentState:
    """
    Detects language and classifies the message as scam or ham using XLM-RoBERTa.
    Populates: language, label, confidence, scam_prob.
    """
    lang   = detect_language(state["message"])
    result = run_classifier(state["message"])
    return {
        **state,
        "language"  : lang,
        "label"     : result["label"],
        "confidence": result["confidence"],
        "scam_prob" : result["scam_prob"],
    }


# ── Conditional routing after Detection Agent ─────────────────────────────────
def route_after_detection(state: AgentState) -> str:
    """Skip Explanation + Risk agents if message is clearly safe (ham ≥ 80% confidence)."""
    if state["label"] == "ham" and state["confidence"] >= 0.80:
        return "safe_exit"
    return "explanation_agent"


# ─────────────────────────────────────────────────────────────────────────────
# Agent 2: Explanation Agent (SHAP + RAG)
# ─────────────────────────────────────────────────────────────────────────────
def explanation_agent(state: AgentState) -> AgentState:
    """
    Uses SHAP to identify suspicious tokens, then retrieves relevant SPF advisory
    passages from the FAISS index to ground the explanation in real SPF data.

    This is the only agent that uses RAG.
    Populates: top_features, rag_chunks, rag_summary.
    """
    # SHAP: which tokens pushed the model toward 'scam'?
    top_features = get_top_shap_features(state["message"], top_n=5)

    # Determine likely scam type early (for targeted RAG retrieval)
    scam_type_guess, _, _, _ = classify_scam_type(state["message"])

    # RAG: retrieve most relevant SPF passages
    rag_chunks = retrieve_spf_passages(state["message"], scam_type_guess, top_k=3)

    # Build a concise RAG-grounded summary from retrieved passages
    key_tokens = ", ".join(f"'{f['token']}'" for f in top_features[:3]) if top_features else "—"
    best_chunk = rag_chunks[0] if rag_chunks else None

    if best_chunk:
        rag_summary = (
            f"Key suspicious tokens: {key_tokens}. "
            f"SPF 2025 (p.{best_chunk['source_page']}): {best_chunk['text'][:220].rstrip()}..."
        )
    else:
        rag_summary = f"Key suspicious tokens: {key_tokens}. No SPF passage retrieved."

    return {
        **state,
        "top_features": top_features,
        "rag_chunks"  : rag_chunks,
        "rag_summary" : rag_summary,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Agent 3: Risk Agent
# ─────────────────────────────────────────────────────────────────────────────
def risk_agent(state: AgentState) -> AgentState:
    """
    Maps the message to the SPF 2025 taxonomy, computes a risk score,
    and selects an appropriate advisory.
    Populates: scam_type, spf_name, risk_score, risk_tier, advice.
    """
    scam_type, base_score, risk_tier, advice = classify_scam_type(state["message"])

    
    adjusted_score = max(10, min(100, int(base_score * state["scam_prob"])))

    if adjusted_score >= 85:
        risk_tier = "CRITICAL"
    elif adjusted_score >= 65:
        risk_tier = "HIGH"
    elif adjusted_score >= 40:
        risk_tier = "MEDIUM"
    else:
        risk_tier = "LOW"

    spf_info = SPF_TAXONOMY.get(scam_type, SPF_TAXONOMY["unknown"])

    output = {
        "message"         : state["message"],
        "language"        : state["language"],
        "label"           : state["label"],
        "confidence"      : state["confidence"],
        "scam_prob"       : state["scam_prob"],
        "top_features"    : state.get("top_features", []),
        "rag_chunks"      : state.get("rag_chunks", []),
        "rag_summary"     : state.get("rag_summary", ""),
        "scam_type"       : scam_type,
        "spf_name"        : spf_info["spf_name"],
        "spf_cases_2025"  : spf_info["2025_cases"],
        "spf_losses_2025" : spf_info["2025_losses"],
        "avg_loss_sgd"    : spf_info["avg_loss"],
        "risk_score"      : adjusted_score,
        "risk_tier"       : risk_tier,
        "advice"          : advice,
    }
    return {**state, "scam_type": scam_type, "spf_name": spf_info["spf_name"],
            "risk_score": adjusted_score, "risk_tier": risk_tier,
            "advice": advice, "output": output}


# ── Safe exit for clearly legitimate messages ─────────────────────────────────
def safe_exit_node(state: AgentState) -> AgentState:
    output = {
        "message"         : state["message"],
        "language"        : state["language"],
        "label"           : "ham",
        "confidence"      : state["confidence"],
        "scam_prob"       : state["scam_prob"],
        "top_features"    : [],
        "rag_chunks"      : [],
        "rag_summary"     : "",
        "scam_type"       : None,
        "spf_name"        : None,
        "spf_cases_2025"  : None,
        "spf_losses_2025" : None,
        "avg_loss_sgd"    : None,
        "risk_score"      : 0,
        "risk_tier"       : "NONE",
        "advice"          : "This message appears legitimate. No action needed.",
    }
    return {**state, "output": output}

print("✅ All 3 agents + safe exit node defined")

✅ All 3 agents + safe exit node defined


## Cell 8 — Assemble LangGraph State Machine

In [8]:
graph = StateGraph(AgentState)

# Register all nodes
graph.add_node("detection_agent",   detection_agent)
graph.add_node("explanation_agent", explanation_agent)
graph.add_node("risk_agent",        risk_agent)
graph.add_node("safe_exit",         safe_exit_node)

# Entry point
graph.set_entry_point("detection_agent")

# Conditional routing: ham with high confidence → skip to safe_exit
graph.add_conditional_edges(
    "detection_agent",
    route_after_detection,
    {
        "safe_exit"        : "safe_exit",
        "explanation_agent": "explanation_agent",
    }
)

# Scam path: Explanation → Risk → END
graph.add_edge("explanation_agent", "risk_agent")
graph.add_edge("risk_agent",        END)
graph.add_edge("safe_exit",         END)

pipeline = graph.compile()

print("✅ LangGraph pipeline compiled")
print("Nodes:", list(pipeline.get_graph().nodes.keys()))

✅ LangGraph pipeline compiled
Nodes: ['__start__', 'detection_agent', 'explanation_agent', 'risk_agent', 'safe_exit', '__end__']


## Cell 9 — Run Helper with Formatted Output

In [9]:
TIER_ICONS = {"CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡", "LOW": "🟢", "NONE": "✅"}
LANG_NAMES = {"en": "English", "zh": "Mandarin", "ta": "Tamil", "ms": "Malay", "singlish": "Singlish"}

def run_pipeline(message: str, verbose: bool = True) -> dict:
    initial: AgentState = {
        "message": message, "language": None, "label": None,
        "confidence": None, "scam_prob": None, "top_features": None,
        "rag_chunks": None, "rag_summary": None, "scam_type": None,
        "spf_name": None, "risk_score": None, "risk_tier": None,
        "advice": None, "output": None,
    }
    final  = pipeline.invoke(initial)
    result = final["output"]

    if verbose:
        tier = result["risk_tier"]
        icon = TIER_ICONS.get(tier, "❓")
        lang = LANG_NAMES.get(result["language"], result["language"])

        sep = "─" * 64
        print(f"\n{sep}")
        print(f"  INPUT   : {message[:80]}")
        print(sep)

        # ── Core prediction table ─────────────────────────────────────────────
        core = pd.DataFrame([{
            "Prediction" : result["label"].upper(),
            "Confidence" : f"{result['confidence']:.1%}",
            "Language"   : lang,
            "Scam Prob"  : f"{result['scam_prob']:.1%}",
        }])
        print(core.to_string(index=False))

        if result["label"] == "scam":
            # ── Risk table ────────────────────────────────────────────────────
            print()
            risk_tbl = pd.DataFrame([{
                "Risk Tier"    : f"{icon} {tier}",
                "Risk Score"   : f"{result['risk_score']}/100",
                "Scam Type"    : result["spf_name"],
                "SPF 2025 Cases" : f"{result['spf_cases_2025']:,}",
                "Total Losses" : f"${result['spf_losses_2025']}M",
                "Avg Loss/Victim": f"${result['avg_loss_sgd']:,}",
            }])
            print(risk_tbl.to_string(index=False))

            # ── SHAP keywords ─────────────────────────────────────────────────
            if result.get("top_features"):
                print()
                shap_tbl = pd.DataFrame([
                    {"Rank": i+1, "Token": f["token"], "SHAP Score": f"{f['shap_value']:+.4f}"}
                    for i, f in enumerate(result["top_features"])
                ])
                print("  🔍 SHAP — Top suspicious tokens:")
                print(shap_tbl.to_string(index=False))

            # ── RAG retrieved passages ────────────────────────────────────────
            if result.get("rag_chunks"):
                print()
                print("  📚 RAG — Retrieved SPF 2025 passages:")
                rag_tbl = pd.DataFrame([{
                    "ID"       : c["id"],
                    "Type"     : c["scam_type"],
                    "Score"    : f"{c['score']:.3f}",
                    "Page"     : c["source_page"],
                    "Topic"    : c["topic"],
                } for c in result["rag_chunks"]])
                print(rag_tbl.to_string(index=False))

            # ── RAG grounded explanation ──────────────────────────────────────
            print()
            print("  💬 Explanation (RAG-grounded):")
            print(f"     {result['rag_summary']}")

            # ── Advice ───────────────────────────────────────────────────────
            print()
            print("  ⚠️  Advice:")
            print(f"     {result['advice']}")

        else:
            print()
            print(f"  ✅ {result['advice']}")

        print(sep)

    return result

print("✅ Run helper ready")

✅ Run helper ready


## Cell 10 — Inference: Seven Test Cases (One per Language & Scam Type)

In [10]:
test_cases = [
    ("en / phishing",
     "Your POSB account has been suspended. Verify your details at http://posb-secure-login.xyz or it will be closed."),
    ("en / investment",
     "Exclusive crypto investment opportunity! Guaranteed 300% returns in 30 days. Join our Telegram group now."),
    ("ms / job scam",
     "Kerja mudah dari rumah! Pendapatan RM5,000 sebulan. Bayar yuran pendaftaran RM200 sahaja untuk mula."),
    ("ta / investment",
     "உங்கள் முதலீட்டில் 300% லாபம் உறுதி. இப்போதே பதிவு செய்யுங்கள், வாய்ப்பு வரையறுக்கப்பட்டது!"),
    ("zh / impersonation",
     "您好，我是新加坡警察局侦探。您的账户涉嫌洗钱，请立即转账至安全账户，否则将被逮捕。"),
    ("singlish / ecommerce",
     "Eh bro selling iPhone 15 Pro Max $400 only lah! Very cheap one. PayNow me first then I ship lor. Confirm legit wan!"),
    ("en / ham",
     "Hi! Reminder: your dentist appointment is tomorrow at 2pm at Raffles Dental. Please reply to confirm."),
]

print("=" * 64)
print("  SCAMSENSE — LangGraph 3-Agent Pipeline: Inference Results")
print("=" * 64)

all_results = []
for name, msg in test_cases:
    print(f"\n▶ TEST CASE: {name.upper()}")
    r = run_pipeline(msg, verbose=True)
    r["test_case"] = name
    all_results.append(r)

print("\n✅ All test cases complete")

  SCAMSENSE — LangGraph 3-Agent Pipeline: Inference Results

▶ TEST CASE: EN / PHISHING


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:41, 41.47s/it]               



────────────────────────────────────────────────────────────────
  INPUT   : Your POSB account has been suspended. Verify your details at http://posb-secure-
────────────────────────────────────────────────────────────────
Prediction Confidence Language Scam Prob
      SCAM     100.0%  English    100.0%

Risk Tier Risk Score     Scam Type SPF 2025 Cases Total Losses Avg Loss/Victim
   🟠 HIGH     78/100 Phishing Scam          6,264       $39.9M          $6,384

  🔍 SHAP — Top suspicious tokens:
 Rank    Token SHAP Score
    1   closed    +0.1260
    2  suspend    +0.0278
    3 account     +0.0277
    4       xy    +0.0092
    5       z     +0.0085

  📚 RAG — Retrieved SPF 2025 passages:
     ID     Type Score  Page                                         Topic
spf_019 phishing 0.350    11                      Phishing contact methods
spf_018 phishing 0.326     7 Phishing tactics — card details and OTP theft
spf_017 phishing 0.199     7                 Phishing scam statistics 2025
spf_

  0%|          | 0/498 [00:00<?, ?it/s]

...

PartitionExplainer explainer: 2it [00:29, 29.67s/it]               



────────────────────────────────────────────────────────────────
  INPUT   : 您好，我是新加坡警察局侦探。您的账户涉嫌洗钱，请立即转账至安全账户，否则将被逮捕。
────────────────────────────────────────────────────────────────
Prediction Confidence Language Scam Prob
      SCAM     100.0% Mandarin    100.0%

 Risk Tier Risk Score                               Scam Type SPF 2025 Cases Total Losses Avg Loss/Victim
🔴 CRITICAL     92/100 Government Officials Impersonation Scam          3,363      $242.9M         $72,229

  🔍 SHAP — Top suspicious tokens:
 Rank Token SHAP Score
    1    涉嫌    +0.1796
    2    您的    +0.1062
    3     将    +0.0748
    4    逮捕    +0.0676
    5    账户    +0.0505

  📚 RAG — Retrieved SPF 2025 passages:
     ID          Type Score  Page                                             Topic
spf_012 impersonation 0.552    18 What Singapore government officials will never do
spf_013 impersonation 0.470    17      Government impersonation — PayNow and crypto
spf_011 impersonation 0.398    17   Government imperson

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:29, 29.51s/it]               


────────────────────────────────────────────────────────────────
  INPUT   : Eh bro selling iPhone 15 Pro Max $400 only lah! Very cheap one. PayNow me first 
────────────────────────────────────────────────────────────────
Prediction Confidence Language Scam Prob
      SCAM      87.1% Singlish     87.1%

Risk Tier Risk Score       Scam Type SPF 2025 Cases Total Losses Avg Loss/Victim
 🟡 MEDIUM     52/100 E-commerce Scam          6,703       $16.7M          $2,503

  🔍 SHAP — Top suspicious tokens:
 Rank Token SHAP Score
    1  400     +0.1957
    2     $    +0.1747
    3   Pay    +0.1693
    4 firm     +0.1578
    5 only     +0.1249

  📚 RAG — Retrieved SPF 2025 passages:
     ID      Type Score  Page                                 Topic
spf_021 ecommerce 0.296     6 E-commerce scam platforms and payment
spf_020 ecommerce 0.287     6       E-commerce scam statistics 2025
spf_022 ecommerce 0.205     6   E-commerce scam — most common items

  💬 Explanation (RAG-grounded):
     Key susp

## Cell 11 — Summary Table

In [11]:
rows = []
for r in all_results:
    top_tok  = r["top_features"][0]["token"] if r.get("top_features") else "—"
    rag_used = len(r.get("rag_chunks", []))
    rows.append({
        "Test Case"    : r["test_case"],
        "Prediction"   : r["label"].upper(),
        "Confidence"   : f"{r['confidence']:.1%}",
        "Language"     : LANG_NAMES.get(r["language"], r["language"]),
        "Scam Type"    : r.get("spf_name") or "—",
        "Risk Tier"    : f"{TIER_ICONS.get(r['risk_tier'],'')} {r['risk_tier']}",
        "Risk Score"   : r["risk_score"],
        "Top SHAP Token": top_tok,
        "RAG Passages" : rag_used,
    })

df_summary = pd.DataFrame(rows)
print("\n" + "=" * 64)
print("  PIPELINE SUMMARY — ALL TEST CASES")
print("=" * 64)
print(df_summary.to_string(index=False))

out_dir = BASE_DIR / "results"
out_dir.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(out_dir / "06_agent_inference_results.csv", index=False)
print(f"\n✅ Summary table saved → results/06_agent_inference_results.csv")


  PIPELINE SUMMARY — ALL TEST CASES
           Test Case Prediction Confidence Language                               Scam Type  Risk Tier  Risk Score Top SHAP Token  RAG Passages
       en / phishing       SCAM     100.0%  English                           Phishing Scam     🟠 HIGH          78         closed             4
     en / investment       SCAM     100.0%  English                         Investment Scam 🔴 CRITICAL          95            now             4
       ms / job scam       SCAM     100.0%    Malay                                Job Scam     🟠 HIGH          80         mudah              4
     ta / investment       SCAM     100.0%    Tamil                         Investment Scam 🔴 CRITICAL          95    செய்யுங்கள்             4
  zh / impersonation       SCAM     100.0% Mandarin Government Officials Impersonation Scam 🔴 CRITICAL          92             涉嫌             3
singlish / ecommerce       SCAM      87.1% Singlish                         E-commerce Scam   🟡 MED

## Cell 12 — Save Artifacts & Push to GitHub

In [12]:
import json

# Save SPF taxonomy 
models_dir = BASE_DIR / "models"
models_dir.mkdir(parents=True, exist_ok=True)

taxonomy_out = {
    k: {ek: ev for ek, ev in v.items() if ek != "keywords"}
    for k, v in SPF_TAXONOMY.items()
}
with open(models_dir / "spf_taxonomy.json", "w") as f:
    json.dump(taxonomy_out, f, indent=2)
print("✅ SPF taxonomy saved → models/spf_taxonomy.json")

# Save SPF corpus JSON for downstream notebooks
import shutil
with open(RAG_DIR / "spf_corpus.json", "w") as f:
    json.dump(SPF_CORPUS, f, indent=2, ensure_ascii=False)
print("✅ SPF corpus saved   → rag/spf_corpus.json")
print("✅ FAISS index already saved → rag/spf_faiss.index")

print("\n" + "=" * 64)
print("  Notebook 04 complete!")
print("  Agent 1 : Detection      (XLM-RoBERTa + language detection)")
print("  Agent 2 : Explanation    (SHAP + FAISS RAG over SPF corpus)")
print("  Agent 3 : Risk Scoring   (SPF 2025 taxonomy + advisory)")
print("  Graph   : LangGraph StateGraph with conditional routing")
print("=" * 64)

✅ SPF taxonomy saved → models/spf_taxonomy.json
✅ SPF corpus saved   → rag/spf_corpus.json
✅ FAISS index already saved → rag/spf_faiss.index

  Notebook 04 complete!
  Agent 1 : Detection      (XLM-RoBERTa + language detection)
  Agent 2 : Explanation    (SHAP + FAISS RAG over SPF corpus)
  Agent 3 : Risk Scoring   (SPF 2025 taxonomy + advisory)
  Graph   : LangGraph StateGraph with conditional routing
